In [0]:
# ==========================================================
# Imports
# ==========================================================

import random
import uuid
from datetime import datetime, timedelta
from dataclasses import dataclass

import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql import Window

from faker import Faker

fake = Faker("pt_BR")

random.seed(42)

In [0]:
# ==========================================================
# Simulation Configuration
# ==========================================================

START_DATE = datetime(2026, 7, 22)

SIMULATION_DAYS = 1

NEW_CUSTOMERS_RATE = 0.02

CRM_PHONE_CHANGE = 0.03
CRM_EMAIL_CHANGE = 0.02
CRM_ADDRESS_CHANGE = 0.04

ERP_NEW_ORDER_RATE = 0.05

WEB_LOGIN_RATE = 0.80
WEB_BROWSER_CHANGE = 0.08
WEB_DEVICE_CHANGE = 0.05

LOYALTY_REDEEM_RATE = 0.10

In [0]:
# ==========================================================
# Paths
# ==========================================================

MASTER_PROFILE_PATH = "/Volumes/poc/master_profile"

STATE_PATH = "/Volumes/poc/state"

LANDING = "/Volumes/landing"

In [0]:
# ==========================================================
# Load Current State
# ==========================================================

master = spark.read.parquet(f"{MASTER_PROFILE_PATH}")

crm = spark.read.parquet(f"{STATE_PATH}/crm")

erp = spark.read.parquet(f"{STATE_PATH}/erp")

web = spark.read.parquet(f"{STATE_PATH}/web")

loyalty = spark.read.parquet(f"{STATE_PATH}/loyalty")

orders = spark.read.parquet(f"{STATE_PATH}/orders")

payments = spark.read.parquet(f"{STATE_PATH}/payments")

In [0]:
# ==========================================================
# Convert State
# ==========================================================

master_pd = master.toPandas()

crm_pd = crm.toPandas()

erp_pd = erp.toPandas()

web_pd = web.toPandas()

loyalty_pd = loyalty.toPandas()

orders_pd = orders.toPandas()

payments_pd = payments.toPandas()

In [0]:
# ==========================================================
# Event Model
# ==========================================================

@dataclass

class Event:

    customer_unique_id: str

    event_type: str

    payload: dict

In [0]:
# ==========================================================
# Daily Events
# ==========================================================

events = []

crm_changed = set()

erp_changed = set()

web_changed = set()

loyalty_changed = set()

new_orders = []

new_payments = []

In [0]:
# ==========================================================
# Generate Daily Events
# ==========================================================

print("=" * 80)
print(f"Generating events for {START_DATE.date()}")
print("=" * 80)

# ----------------------------------------------------------
# NEW CUSTOMERS
# ----------------------------------------------------------

new_customer_count = int(len(master_pd) * NEW_CUSTOMERS_RATE)

for _ in range(new_customer_count):

    events.append(

        Event(

            customer_unique_id=None,

            event_type="NEW_CUSTOMER",

            payload={}

        )

    )

# ----------------------------------------------------------
# PHONE CHANGES
# ----------------------------------------------------------

phone_sample = master_pd.sample(
    frac=CRM_PHONE_CHANGE,
    random_state=random.randint(1,100000)
)

for _, customer in phone_sample.iterrows():

    events.append(

        Event(

            customer_unique_id=customer.customer_unique_id,

            event_type="PHONE_CHANGED",

            payload={}

        )

    )

# ----------------------------------------------------------
# EMAIL CHANGES
# ----------------------------------------------------------

email_sample = master_pd.sample(
    frac=CRM_EMAIL_CHANGE,
    random_state=random.randint(1,100000)
)

for _, customer in email_sample.iterrows():

    events.append(

        Event(

            customer_unique_id=customer.customer_unique_id,

            event_type="EMAIL_CHANGED",

            payload={}

        )

    )

# ----------------------------------------------------------
# ADDRESS CHANGES
# ----------------------------------------------------------

address_sample = master_pd.sample(
    frac=CRM_ADDRESS_CHANGE,
    random_state=random.randint(1,100000)
)

for _, customer in address_sample.iterrows():

    events.append(

        Event(

            customer_unique_id=customer.customer_unique_id,

            event_type="ADDRESS_CHANGED",

            payload={}

        )

    )

# ----------------------------------------------------------
# NEW ORDERS
# ----------------------------------------------------------

buyers = master_pd.sample(
    frac=ERP_NEW_ORDER_RATE,
    random_state=random.randint(1,100000)
)

for _, customer in buyers.iterrows():

    amount = round(

        random.uniform(
            40,
            1200
        ),

        2

    )

    events.append(

        Event(

            customer_unique_id=customer.customer_unique_id,

            event_type="NEW_ORDER",

            payload={

                "amount": amount

            }

        )

    )

# ----------------------------------------------------------
# LOGIN
# ----------------------------------------------------------

logins = master_pd.sample(
    frac=WEB_LOGIN_RATE,
    random_state=random.randint(1,100000)
)

for _, customer in logins.iterrows():

    events.append(

        Event(

            customer_unique_id=customer.customer_unique_id,

            event_type="LOGIN",

            payload={}

        )

    )

# ----------------------------------------------------------
# LOYALTY REDEMPTION
# ----------------------------------------------------------

redeemers = master_pd.sample(
    frac=LOYALTY_REDEEM_RATE,
    random_state=random.randint(1,100000)
)

for _, customer in redeemers.iterrows():

    events.append(

        Event(

            customer_unique_id=customer.customer_unique_id,

            event_type="LOYALTY_REDEEM",

            payload={}

        )

    )

random.shuffle(events)

print(f"Events generated: {len(events):,}")

In [0]:
# ==========================================================
# Process Daily Events
# ==========================================================

print("=" * 80)
print("Applying Events...")
print("=" * 80)

for event in events:

    customer = event.customer_unique_id

    # ======================================================
    # NEW CUSTOMER
    # ======================================================

    if event.event_type == "NEW_CUSTOMER":

        #
        # This will be implemented in the next cell.
        # It creates a new customer in the Master Profile,
        # then derives CRM, ERP, WEB and LOYALTY records.
        #

        continue

    # ======================================================
    # PHONE
    # ======================================================

    elif event.event_type == "PHONE_CHANGED":

        idx = crm_pd.customer_unique_id == customer

        crm_pd.loc[idx, "phone"] = fake.phone_number()

        crm_changed.add(customer)

    # ======================================================
    # EMAIL
    # ======================================================

    elif event.event_type == "EMAIL_CHANGED":

        idx = crm_pd.customer_unique_id == customer

        crm_pd.loc[idx, "email"] = fake.email()

        crm_changed.add(customer)

    # ======================================================
    # ADDRESS
    # ======================================================

    elif event.event_type == "ADDRESS_CHANGED":

        idx = crm_pd.customer_unique_id == customer

        crm_pd.loc[idx, "street"] = fake.street_address()

        crm_pd.loc[idx, "city"] = fake.city()

        crm_pd.loc[idx, "state"] = fake.estado_sigla()

        crm_changed.add(customer)

    # ======================================================
    # NEW ORDER
    # ======================================================

    elif event.event_type == "NEW_ORDER":

        amount = event.payload["amount"]

        order_id = str(uuid.uuid4())

        payment_id = str(uuid.uuid4())

        order = {

            "order_id": order_id,

            "customer_unique_id": customer,

            "order_purchase_timestamp": START_DATE,

            "order_status": "delivered"

        }

        payment = {

            "payment_id": payment_id,

            "order_id": order_id,

            "payment_value": amount

        }

        new_orders.append(order)

        new_payments.append(payment)

        #
        # Update ERP totals
        #

        idx = erp_pd.customer_unique_id == customer

        erp_pd.loc[idx, "total_orders"] += 1

        erp_pd.loc[idx, "total_spent"] += amount

        erp_changed.add(customer)

        #
        # Loyalty also changes
        #

        loyalty_changed.add(customer)

    # ======================================================
    # LOGIN
    # ======================================================

    elif event.event_type == "LOGIN":

        idx = web_pd.customer_unique_id == customer

        web_pd.loc[idx, "last_login"] = datetime.now()

        if random.random() < WEB_BROWSER_CHANGE:

            web_pd.loc[idx, "browser"] = random.choice(BROWSERS)

        if random.random() < WEB_DEVICE_CHANGE:

            web_pd.loc[idx, "device"] = random.choice(DEVICES)

        web_changed.add(customer)

    # ======================================================
    # LOYALTY
    # ======================================================

    elif event.event_type == "LOYALTY_REDEEM":

        idx = loyalty_pd.customer_unique_id == customer

        available = loyalty_pd.loc[idx, "available_points"].iloc[0]

        redeemed = min(

            available,

            random.randint(
                100,
                5000
            )

        )

        loyalty_pd.loc[idx, "available_points"] -= redeemed

        loyalty_pd.loc[idx, "redeemed_points"] += redeemed

        loyalty_changed.add(customer)

print()

print("CRM changed:", len(crm_changed))

print("ERP changed:", len(erp_changed))

print("WEB changed:", len(web_changed))

print("LOYALTY changed:", len(loyalty_changed))

print("New Orders:", len(new_orders))

In [0]:
# ==========================================================
# Apply NEW CUSTOMER Events
# ==========================================================

print("=" * 80)
print("Creating New Customers...")
print("=" * 80)

new_master_rows = []
new_crm_rows = []
new_erp_rows = []
new_web_rows = []
new_loyalty_rows = []

current_customer = len(master_pd) + 1

for event in events:

    if event.event_type != "NEW_CUSTOMER":
        continue

    # ------------------------------------------------------
    # Master Profile
    # ------------------------------------------------------

    customer_unique_id = str(uuid.uuid4())

    first_name = fake.first_name()

    middle_name = fake.first_name() if random.random() < 0.40 else None

    last_name = fake.last_name()

    full_name = " ".join(
        x for x in [first_name, middle_name, last_name]
        if x is not None
    )

    phone = fake.phone_number()

    email = random_email(
        first_name,
        last_name
    )

    street = fake.street_address()

    city = fake.city()

    state = fake.estado_sigla()

    cpf = fake.cpf()

    birth_date = fake.date_of_birth(
        minimum_age=18,
        maximum_age=80
    )

    global_customer_id = f"CUST{current_customer:08d}"

    master_row = {

        "global_customer_id": global_customer_id,

        "customer_unique_id": customer_unique_id,

        "first_name": first_name,

        "middle_name": middle_name,

        "last_name": last_name,

        "cpf": cpf,

        "birth_date": birth_date,

        "phone": phone,

        "personal_email": email,

        "street": street,

        "city": city,

        "state": state

    }

    new_master_rows.append(master_row)

    # ------------------------------------------------------
    # CRM
    # ------------------------------------------------------

    crm_row = {

        "crm_customer_id": f"CRM{current_customer:08d}",

        "customer_unique_id": customer_unique_id,

        "first_name": first_name,

        "middle_name": middle_name,

        "last_name": last_name,

        "full_name": full_name,

        "phone": phone,

        "email": email,

        "street": street,

        "city": city,

        "state": state,

        "profession": fake.job(),

        "status": "ACTIVE",

        "risk_score": random.randint(0,100)

    }

    new_crm_rows.append(crm_row)

    crm_changed.add(customer_unique_id)

    # ------------------------------------------------------
    # ERP
    # ------------------------------------------------------

    erp_row = {

        "erp_customer_id": f"ERP{current_customer:08d}",

        "customer_unique_id": customer_unique_id,

        "customer_name": f"{last_name}, {first_name}",

        "total_orders": 0,

        "total_spent": 0.0,

        "average_ticket": 0.0,

        "credit_limit": 3000.0,

        "customer_group": "Retail",

        "payment_terms": "NET15"

    }

    new_erp_rows.append(erp_row)

    erp_changed.add(customer_unique_id)

    # ------------------------------------------------------
    # Ecommerce
    # ------------------------------------------------------

    web_row = {

        "ecommerce_customer_id": f"WEB{current_customer:08d}",

        "customer_unique_id": customer_unique_id,

        "username": fake.user_name(),

        "display_name": random_name(
            first_name,
            middle_name,
            last_name
        ),

        "email_address": email,

        "phone_number": phone,

        "browser": random.choice(BROWSERS),

        "operating_system": random.choice(
            OPERATING_SYSTEMS
        ),

        "device": random.choice(
            DEVICES
        ),

        "newsletter_optin": random.random() < 0.70,

        "preferred_category": None,

        "wishlist_items": 0,

        "abandoned_carts": 0,

        "account_created": START_DATE,

        "last_login": START_DATE

    }

    new_web_rows.append(web_row)

    web_changed.add(customer_unique_id)

    # ------------------------------------------------------
    # Loyalty
    # ------------------------------------------------------

    loyalty_row = {

        "loyalty_customer_id": f"LOY{current_customer:08d}",

        "customer_unique_id": customer_unique_id,

        "member_name": full_name,

        "join_date": START_DATE,

        "points": 0,

        "redeemed_points": 0,

        "available_points": 0,

        "tier": "Bronze",

        "wallet_balance": 0.0,

        "active_member": True

    }

    new_loyalty_rows.append(loyalty_row)

    loyalty_changed.add(customer_unique_id)

    current_customer += 1

# ----------------------------------------------------------
# Append to current state
# ----------------------------------------------------------

master_pd = pd.concat(
    [master_pd, pd.DataFrame(new_master_rows)],
    ignore_index=True
)

crm_pd = pd.concat(
    [crm_pd, pd.DataFrame(new_crm_rows)],
    ignore_index=True
)

erp_pd = pd.concat(
    [erp_pd, pd.DataFrame(new_erp_rows)],
    ignore_index=True
)

web_pd = pd.concat(
    [web_pd, pd.DataFrame(new_web_rows)],
    ignore_index=True
)

loyalty_pd = pd.concat(
    [loyalty_pd, pd.DataFrame(new_loyalty_rows)],
    ignore_index=True
)

print(f"New customers created: {len(new_master_rows):,}")

In [0]:
# ==========================================================
# Recalculate Business Metrics
# ==========================================================

print("=" * 80)
print("Recalculating Business Metrics...")
print("=" * 80)

#
# ERP
#

for customer in erp_changed:

    idx = erp_pd.customer_unique_id == customer

    if idx.sum() == 0:
        continue

    total_orders = int(
        erp_pd.loc[idx, "total_orders"].iloc[0]
    )

    total_spent = float(
        erp_pd.loc[idx, "total_spent"].iloc[0]
    )

    #
    # Average Ticket
    #

    average_ticket = (

        total_spent / total_orders

        if total_orders > 0

        else 0

    )

    #
    # Customer Group
    #

    group = customer_group(
        total_spent
    )

    #
    # Credit Limit
    #

    credit = calculate_credit_limit(

        total_spent,

        total_orders

    )

    #
    # Payment Terms
    #

    payment = payment_terms(
        group
    )

    #
    # Account Manager
    #

    manager = choose_manager(
        group
    )

    #
    # Update ERP
    #

    erp_pd.loc[idx, "average_ticket"] = average_ticket

    erp_pd.loc[idx, "customer_group"] = group

    erp_pd.loc[idx, "credit_limit"] = credit

    erp_pd.loc[idx, "payment_terms"] = payment

    erp_pd.loc[idx, "account_manager"] = manager


#
# Loyalty
#

for customer in loyalty_changed:

    idx = loyalty_pd.customer_unique_id == customer

    if idx.sum() == 0:
        continue

    #
    # Get ERP Total Spent
    #

    erp_idx = erp_pd.customer_unique_id == customer

    total_spent = float(

        erp_pd.loc[
            erp_idx,
            "total_spent"
        ].iloc[0]

    )

    #
    # Earned Points
    #

    earned_points = calculate_points(
        total_spent
    )

    redeemed = int(

        loyalty_pd.loc[
            idx,
            "redeemed_points"
        ].iloc[0]

    )

    available = max(

        earned_points - redeemed,

        0

    )

    tier = loyalty_tier(
        earned_points
    )

    wallet = wallet_balance(
        earned_points
    )

    #
    # Update Loyalty
    #

    loyalty_pd.loc[idx, "points"] = earned_points

    loyalty_pd.loc[idx, "available_points"] = available

    loyalty_pd.loc[idx, "tier"] = tier

    loyalty_pd.loc[idx, "wallet_balance"] = wallet

print()

print(f"ERP recalculated: {len(erp_changed):,}")

print(f"Loyalty recalculated: {len(loyalty_changed):,}")

In [0]:
# ==========================================================
# Build Incrementals
# ==========================================================

print("=" * 80)
print("Building Incremental Datasets...")
print("=" * 80)

#
# CRM
#

crm_incremental = crm_pd[
    crm_pd.customer_unique_id.isin(
        change_tracker["crm"]
    )
].copy()

#
# ERP Customers
#

erp_incremental = erp_pd[
    erp_pd.customer_unique_id.isin(
        change_tracker["erp"]
    )
].copy()

#
# Ecommerce
#

web_incremental = web_pd[
    web_pd.customer_unique_id.isin(
        change_tracker["web"]
    )
].copy()

#
# Loyalty
#

loyalty_incremental = loyalty_pd[
    loyalty_pd.customer_unique_id.isin(
        change_tracker["loyalty"]
    )
].copy()

#
# Orders
#

orders_incremental = pd.DataFrame(
    change_tracker["orders"]
)

# Publish the ERP key instead of the simulator-only customer key.
if not orders_incremental.empty:
    orders_incremental = (
        orders_incremental
        .merge(
            erp_pd[["customer_unique_id", "erp_customer_id"]],
            on="customer_unique_id",
            how="left",
            validate="many_to_one"
        )
        .drop(columns=["customer_unique_id"])
    )

#
# Payments
#

payments_incremental = pd.DataFrame(
    change_tracker["payments"]
)

print()

print(f"CRM        : {len(crm_incremental):,}")

print(f"ERP        : {len(erp_incremental):,}")

print(f"WEB        : {len(web_incremental):,}")

print(f"LOYALTY    : {len(loyalty_incremental):,}")

print(f"ORDERS     : {len(orders_incremental):,}")

print(f"PAYMENTS   : {len(payments_incremental):,}")

In [0]:
# ==========================================================
# Convert Incrementals to Spark
# ==========================================================

crm_incremental = spark.createDataFrame(
    crm_incremental.drop(columns=["customer_unique_id"])
)

erp_incremental = spark.createDataFrame(
    erp_incremental.drop(columns=["customer_unique_id"])
)

web_incremental = spark.createDataFrame(
    web_incremental.drop(columns=["customer_unique_id"])
)

loyalty_incremental = spark.createDataFrame(
    loyalty_incremental.drop(columns=["customer_unique_id"])
)

orders_incremental = spark.createDataFrame(
    orders_incremental
)

payments_incremental = spark.createDataFrame(
    payments_incremental
)

In [0]:
# ==========================================================
# Landing Date
# ==========================================================

landing_date = START_DATE.strftime("%Y/%m/%d")

print(landing_date)

In [0]:
# ==========================================================
# Write Landing
# ==========================================================

def repair_utf8_mojibake(df):
    """Repair text decoded as Latin-1 while preserving valid UTF-8 values."""
    markers = "|".join(chr(codepoint) for codepoint in (0x00C3, 0x00C2, 0x00E2))

    for field in df.schema.fields:
        if field.dataType.simpleString() != "string":
            continue

        value = F.col(field.name)
        repaired = F.decode(F.encode(value, "ISO-8859-1"), "UTF-8")
        df = df.withColumn(
            field.name,
            F.when(value.rlike(markers), repaired).otherwise(value)
        )

    return df

datasets = {

    f"{LANDING}/crm/customers/{landing_date}":
        crm_incremental,

    f"{LANDING}/erp/customers/{landing_date}":
        erp_incremental,

    f"{LANDING}/erp/orders/{landing_date}":
        orders_incremental,

    f"{LANDING}/erp/payments/{landing_date}":
        payments_incremental,

    f"{LANDING}/ecommerce/customers/{landing_date}":
        web_incremental,

    f"{LANDING}/loyalty/customers/{landing_date}":
        loyalty_incremental

}

for path, df in datasets.items():

    print(path)

    clean_df = repair_utf8_mojibake(df)

    (
        clean_df.write
        .mode("overwrite")
        .parquet(path)
    )

In [0]:
# ==========================================================
# Persist Simulation State
# ==========================================================

print("=" * 80)
print("Persisting Simulation State...")
print("=" * 80)

repair_utf8_mojibake(spark.createDataFrame(master_pd)) \
    .write \
    .mode("overwrite") \
    .parquet(f"{STATE_PATH}/master")

repair_utf8_mojibake(spark.createDataFrame(crm_pd)) \
    .write \
    .mode("overwrite") \
    .parquet(f"{STATE_PATH}/crm")

repair_utf8_mojibake(spark.createDataFrame(erp_pd)) \
    .write \
    .mode("overwrite") \
    .parquet(f"{STATE_PATH}/erp")

repair_utf8_mojibake(spark.createDataFrame(web_pd)) \
    .write \
    .mode("overwrite") \
    .parquet(f"{STATE_PATH}/web")

repair_utf8_mojibake(spark.createDataFrame(loyalty_pd)) \
    .write \
    .mode("overwrite") \
    .parquet(f"{STATE_PATH}/loyalty")

spark.createDataFrame(
    pd.concat(
        [
            orders_pd,
            pd.DataFrame(change_tracker["orders"])
        ],
        ignore_index=True
    )
).write.mode("overwrite").parquet(f"{STATE_PATH}/orders")

spark.createDataFrame(
    pd.concat(
        [
            payments_pd,
            pd.DataFrame(change_tracker["payments"])
        ],
        ignore_index=True
    )
).write.mode("overwrite").parquet(f"{STATE_PATH}/payments")

print("State updated.")

In [0]:
# ==========================================================
# Simulation Control
# ==========================================================

control = pd.DataFrame([

    {

        "last_execution": START_DATE,

        "next_execution": START_DATE + timedelta(days=1),

        "crm_records": len(crm_pd),

        "erp_records": len(erp_pd),

        "web_records": len(web_pd),

        "loyalty_records": len(loyalty_pd),

        "orders": len(orders_pd) + len(change_tracker["orders"]),

        "payments": len(payments_pd) + len(change_tracker["payments"])

    }

])

spark.createDataFrame(control) \
    .write \
    .mode("overwrite") \
    .parquet(f"{STATE_PATH}/simulation_control")

In [0]:
# ==========================================================
# Summary
# ==========================================================

summary = pd.DataFrame({

    "Dataset": [

        "CRM",

        "ERP",

        "WEB",

        "LOYALTY",

        "ORDERS",

        "PAYMENTS"

    ],

    "Incremental Rows":[

        len(crm_incremental),

        len(erp_incremental),

        len(web_incremental),

        len(loyalty_incremental),

        len(orders_incremental),

        len(payments_incremental)

    ]

})

display(summary)